# Aquisição da imagem Sentinel-2 (estágio 01)

Carrega a malha vetorial do IBGE, define o polígono da Região Geográfica Imediata de Guaxupé (310044) como área de estudo (AOI) e adquire o mosaico livre de nuvens do Sentinel-2 Level-2A (bandas B2, B3, B4 e B8) via Google Earth Engine, exportando o resultado em GeoTIFF para o armazenamento canônico em `MyDrive/tcc/`.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Instala as versões fixadas em `requirements-runtime.txt` (incluindo geopandas, usado na leitura da malha vetorial), garantindo o mesmo conjunto de bibliotecas nas duas plataformas.

In [ ]:
# Instala as versões pinadas do requirements-runtime.txt no ambiente atual.
import subprocess
import sys

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)
print("Dependências instaladas a partir de:", requirements)

## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`.

In [ ]:
# Resolve o contexto de runtime: plataforma, armazenamento canônico e configuração.
from src import io
from src.setup import initialize

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
print(f"Plataforma: {platform}")
print(f"Mosaico (raw Sentinel-2): {storage_paths['data_raw_sentinel2']}")
print(f"Figuras: {storage_paths['artifacts_figures']}")

## Autenticação do Earth Engine

Autentica e inicializa o GEE com a conta principal: fluxo interativo no Colab; no Kaggle, carrega a credencial da variável `GEE_CREDENTIALS`. A lógica de plataforma fica isolada em `src/`.

In [ ]:
# Autentica e inicializa o Earth Engine com a conta principal.
from src.utils import authenticate_gee

authenticate_gee()
print("Earth Engine autenticado e inicializado.")

## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, além de registrar as versões dos pacotes principais do ambiente, garantindo o mesmo protocolo de execução nas duas plataformas.

In [ ]:
# Fixa sementes, flags determinísticas e registra as versões do ambiente.
from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config)
log_environment()

## Área de estudo (AOI) a partir da malha IBGE

Lê a malha vetorial das Regiões Geográficas Imediatas de Minas Gerais (dado de referência versionado em `data/external/`), filtra a região `310044` (Guaxupé), reprojeta para EPSG:4326 e converte o polígono em geometria do Earth Engine.

In [ ]:
# Carrega a malha IBGE, filtra a RGI 310044 e converte o AOI para geometria do GEE.
from src.data import gee_client

mesh_path = workspace / config["aoi"]["mesh_path"]
aoi_geometry, aoi_gdf = gee_client.load_aoi_geometry(
    mesh_path, config["aoi"]["region_code"]
)
aoi_row = aoi_gdf.iloc[0]
print(f"AOI: {aoi_row['NM_RGI']} (CD_RGI={aoi_row['CD_RGI']})")
print(f"Área: {float(aoi_row['AREA_KM2']):,.1f} km²")
print(f"Extensão (EPSG:4326): {list(aoi_gdf.total_bounds)}")

## Coleção Sentinel-2 filtrada

Constrói a coleção `COPERNICUS/S2_SR_HARMONIZED` limitada ao AOI e ao período definido em `src/config.yaml`, descarta cenas com mais de 20% de cobertura de nuvens e mascara nuvens e sombras pela banda QA60 (bits 10 e 11), mantendo apenas as bandas B2, B3, B4 e B8.

In [ ]:
# Constrói a coleção Sentinel-2 filtrada por AOI, datas e cobertura de nuvem.
from src.data.gee_client import build_s2_collection

data_cfg = config["data"]
collection = build_s2_collection(
    aoi=aoi_geometry,
    bands=data_cfg["bands"],
    start=data_cfg["dates"]["start"],
    end=data_cfg["dates"]["end"],
    cloud_threshold=data_cfg["cloud_threshold"],
)
print(f"Cenas disponíveis no período: {collection.size().getInfo()}")

## Mosaico anual sem nuvens

Compõe o mosaico mediano de todas as cenas do período e recorta ao polígono do AOI, produzindo a imagem final com as bandas espectrais em resolução nativa de 10 m.

In [ ]:
# Compõe o mosaico mediano do período e recorta ao polígono do AOI.
from src.data.gee_client import build_annual_mosaic

mosaic = build_annual_mosaic(collection, aoi_geometry)
print(f"Bandas do mosaico: {mosaic.bandNames().getInfo()}")

## Exportação do mosaico em GeoTIFF

Verifica se o GeoTIFF já existe no caminho canônico (execuções repetidas não sobrescrevem artefatos). Como o GEE só exporta para uma pasta única na raiz do Drive (o parâmetro `folder` não aceita subcaminhos), o mosaico é exportado para a pasta `tcc` e, após a conclusão da tarefa, realocado para `MyDrive/tcc/data/raw/sentinel2/` no datum SIRGAS 2000 / UTM 23S (resolvido do `src/config.yaml`).

In [ ]:
# Garante o mosaico Sentinel-2 exportado no caminho canônico (reutiliza se já existir).
from src.data.gee_client import ensure_mosaic_export

target_path, exported_now = ensure_mosaic_export(mosaic, storage_paths)

## Pré-visualização do mosaico

Gera uma miniatura RGB do mosaico (bandas B4, B3, B2) via Earth Engine e salva a figura em `MyDrive/tcc/artifacts/figures/` como registro visual da etapa.

In [ ]:
# Salva a miniatura RGB do mosaico no diretório de figuras (idempotente).
from src.data.gee_client import save_mosaic_preview

figure_path = save_mosaic_preview(mosaic, storage_paths)

## Resumo da etapa

Exibe o resumo da aquisição: AOI definido, cenas consideradas, mosaico produzido e caminho do artefato exportado.

In [ ]:
# Exibe o resumo da etapa de aquisição.
summary = {
    "AOI": aoi_row["NM_RGI"],
    "Cenas no período": collection.size().getInfo(),
    "Bandas do mosaico": mosaic.bandNames().getInfo(),
    "Mosaico exportado": str(target_path),
    "Situação": "exportado nesta execução" if exported_now else "pré-existente",
}
from src.utils import print_summary

print_summary(summary, "01")